# core

> Claude code api backend for fastllm

Claude Code has no stateless chat endpoint, but a saved transcript plus `resume` behaves like one. Each request files the message history as a synthetic session (`llmsurgery.ant.msgs2sess`) and resumes it; fastllm's tools are offered through Claude Code's native deferral mechanism (`defer_tools`), so Claude requests calls, the run stops, and fastllm executes them; a continuation feeds the results back through the same tool channel (`hold_result`/`mk_deferred`); and the reply streams back as raw Anthropic events (`aquery_events`), which the standard `fastllm.anthropic` normalizers already understand.

In [ ]:
#| default_exp core

In [ ]:
from fastcore.utils import *
from claude_agent_sdk import ClaudeAgentOptions
from fastllm.types import *
from fastllm.anthropic import (norm_sse_event, norm_tool_calls, norm_parts,
    norm_finish, norm_usage, finalize_usage, denorm_msgs, delta_index_fn, cost)
from fastllm.streaming import mk_acollect_stream
from fastspec.errors import APIError
from llmsurgery.ant import *

Synthetic sessions are filed under one dedicated project directory, keeping resumed histories out of real projects. `SERVER_TOOLS` are the tools Claude Code executes itself—server-side from fastllm's point of view—and keep their bare names in transcripts; every other tool is hosted under `MCP_PREFIX`. `CONT_PROMPT` is the wording Claude Code itself inserts when a turn continues without new user input; it serves the rare final turns that carry neither text nor tool results.

In [ ]:
MCP_SERVER_NAME = "fastllm"
MCP_PREFIX = f"mcp__{MCP_SERVER_NAME}__"
SERVER_TOOLS = ["WebSearch", "WebFetch"]
CONT_PROMPT = "Continue from where you left off."
WORK_DIR = Path.home() / ".fastllm-claude-agent"
WORK_DIR.mkdir(exist_ok=True)

The transcripts land where any Claude Code project's would:

In [ ]:
sess_dir(WORK_DIR)

Path('/Users/jhoward/.claude/projects/-Users-jhoward--fastllm-claude-agent')

`claude_mk_payload` splits the conversation by its final message. A user message of text becomes the query's prompt, with everything before it filed and resumed. A message of tool results is a continuation: all results but the last are filed as ordinary records, the last call is marked deferred (`mk_deferred`) with its result held (`hold_result`), and `no_prompt()` resumes without adding a user turn—Claude Code re-invokes the call, receives the held result, and continues behind its own sentinel. Anything else—a mixed or empty final turn—is filed whole behind `CONT_PROMPT`. Identical requests map to the same session file, and the blanked `ANTHROPIC_API_KEY` keeps the spawned Claude Code on its own login rather than an inherited key.

In [ ]:
def _find_tu(den, tid):
    "The `tool_use` block in `den` with id `tid`"
    return first(b for m in reversed(den) if isinstance(m.get('content'), list)
        for b in m['content'] if b.get('type')=='tool_use' and b.get('id')==tid)

def claude_mk_payload(msgs, model, stream=False, **kwargs):
    "Build prompt + options for a Claude Code SDK `query`: file the past, then a real user turn or a deferred continuation."
    system, tools = kwargs.get('system'), kwargs.get('tools')
    triples = [s for t in (tools or []) if (s:=fn_schema(t)) and s[0]]
    den = prefix_tools(denorm_msgs(msgs), MCP_PREFIX, skip=SERVER_TOOLS)
    held, extra, prompt = {}, [], CONT_PROMPT
    if den and den[-1]['role']=='user':
        blocks = den[-1]['content']
        trs = [b for b in blocks if b.get('type')=='tool_result']
        txt = '\n'.join(b.get('text','') for b in blocks if b.get('type')=='text')
        if txt: den, prompt = den[:-1], txt
        elif trs:
            tu = _find_tu(den[:-1], trs[-1].get('tool_use_id'))
            if tu and any(tu['name']==f'{MCP_PREFIX}{nm}' for nm,_,_ in triples):
                hold_result(held, tu['name'], tu.get('input',{}), trs[-1].get('content',''), trs[-1].get('is_error',False))
                extra = [mk_deferred(tu, cwd=WORK_DIR, uid=stable_uuid(f'{MCP_SERVER_NAME}-deferred:{tu["id"]}'))]
                rest = [b for b in blocks if b is not trs[-1]]
                den = den[:-1] + ([dict(den[-1], content=rest)] if rest else [])
                prompt = no_prompt()
    mcp_servers, allowed, hooks = defer_tools(MCP_SERVER_NAME, triples, held)
    opt_kw = dict(model=model, env={'ANTHROPIC_API_KEY': ''}, cwd=str(WORK_DIR), include_partial_messages=True,
        permission_mode="default", system_prompt=system or "", mcp_servers=mcp_servers, allowed_tools=allowed,
        hooks=hooks or None, strict_mcp_config=True, tools=SERVER_TOOLS if kwargs.get('web_search_options') is not None else [])
    if den or extra:
        opt_kw['resume'] = msgs2sess(den, key='fastllm-claude-code', cwd=WORK_DIR, extra=extra, model=model, entrypoint='sdk-py')
    return dict(prompt=prompt, options=ClaudeAgentOptions(**opt_kw))

In [ ]:
from fastllm.chat import mk_msgs, acomplete, lite_mk_func, AsyncChat
from fastcore.test import *

In [ ]:
def simple_add(a: int, b: int) -> int:
    "Add two numbers"
    return a + b

A short history with one tool shows the request pieces: the new turn as the prompt, the hosted tool's permission rule, and the deterministic session id.

In [ ]:
msgs = mk_msgs(['What is 2+2?', 'It is 4.', 'And 3+5?'])
p = claude_mk_payload(msgs, 'claude-sonnet-4-6', tools=[lite_mk_func(simple_add)])
opts = p['options']
test_eq(p['prompt'], 'And 3+5?')
test_eq(opts.allowed_tools, ['mcp__fastllm__simple_add'])
test_eq(claude_mk_payload(msgs, 'claude-sonnet-4-6')['options'].resume, opts.resume)
opts.resume

'456c232a-40cb-5f9f-86cb-ef35d07df0b8'

The filed history reads back as an ordinary transcript:

In [ ]:
print(show_recs(load_sess(opts.resume, WORK_DIR)))

--- user 2026-01-01T00:00:00 ---
What is 2+2?
--- assistant 2026-01-01T00:00:00 ---
It is 4.


A continuation—the final message holding one tool result—files the history only up to the call, marks the call deferred, and holds the result out of the file entirely: it will arrive through the tool channel when Claude Code re-invokes the call. The prompt adds nothing. With several parallel results, all but the last are filed as ordinary records and only the last rides the tool channel.

In [ ]:
cmsgs = mk_msgs(['What is 7+3? Use the tool.']) + [
    Msg(role='assistant', content=[Part(type=PartType.tool_use, data=dict(id='toolu_01', name='simple_add', arguments=dict(a=7,b=3)))]),
    Msg(role='tool', content=[Part(type=PartType.tool_result, data=dict(id='toolu_01'), text='10')])]
cp = claude_mk_payload(cmsgs, 'claude-sonnet-4-6', tools=[lite_mk_func(simple_add)])
assert not isinstance(cp['prompt'], str)
crecs = load_sess(cp['options'].resume, WORK_DIR)
test_eq(crecs[-1]['attachment']['toolName'], 'mcp__fastllm__simple_add')
test_eq(crecs[-1]['attachment']['toolInput'], dict(a=7, b=3))
print(show_recs(crecs))

--- user 2026-01-01T00:00:00 ---
What is 7+3? Use the tool.
--- assistant 2026-01-01T00:00:00 ---
mcp__fastllm__simple_add


`aquery_events` does the Claude-Code-specific streaming work—re-indexing and bare tool names—and a deferred run ends by itself with a complete stream, so what's left is translation: normalize each event to a fastllm delta, mark `SERVER_TOOLS` calls as server-executed, and surface a `QueryError` as fastllm's `APIError`.

In [ ]:
async def claude_acollect_stream(payload, **kwargs):
    opts, prompt = payload["options"], payload["prompt"]
    async def _gen():
        try:
            async for ev in aquery_events(prompt, opts, prefix=MCP_PREFIX):
                delta = norm_sse_event(ev)
                for tc in (delta.tool_calls or []):
                    if tc.name in SERVER_TOOLS: tc.server = True
                yield delta
        except QueryError as e: raise APIError(str(e), provider='claude_code', model=opts.model, status_code=e.status, raw=e.result)
    async for o in mk_acollect_stream(_gen(), index_fn=delta_index_fn, api_name='claude_code', **kwargs): yield o

Registration reuses the Anthropic normalizers wholesale; the whole backend is the two functions above.

In [ ]:
api_registry.register('claude_code',
    norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    finalize_usage=finalize_usage, mk_payload=claude_mk_payload, acollect_stream=claude_acollect_stream, cost=cost)

### Tests

In [ ]:
msgs = mk_msgs("What is 2+2?")
r = await acomplete(msgs, 'claude-sonnet-4-6', api_name='claude_code', stream=True)
async for o in r:
    if isinstance(o, Completion): print(f"\n--- finish: {o.finish_reason}, usage: {o.usage}")
    elif t := o.get('text'): print(t, end='')

4
--- finish: stop, usage: Usage(prompt_tokens=1091, completion_tokens=5, total_tokens=1096, cached_tokens=1088, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 3, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 1088, 'output_tokens': 5, 'output_tokens_details': {'thinking_tokens': 0}, 'iterations': [{'input_tokens': 3, 'output_tokens': 5, 'cache_read_input_tokens': 1088, 'cache_creation_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}]})


In [ ]:
msgs = mk_msgs("What is 3+5 and 10+5? Use the simple_add tool in parallel.")
r = await acomplete(msgs, 'claude-sonnet-4-6', api_name='claude_code', stream=True, tools=[lite_mk_func(simple_add)])

async for o in r:
    if isinstance(o, Completion):
        print(f"\n--- finish: {o.finish_reason}")
        print(f"--- tool_calls: {o.tool_calls}")
    elif isinstance(o, Part): print(f"\n[Part {o.type}: {o.data}]")
    elif t := o.get('text'): print(t, end='')

Sure! I'll calculate both simultaneously!
[Part tool_use: {'caller': {'type': 'direct'}, 'id': 'toolu_014xhQQjAj2fjzHeakyKgw8y', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}]

[Part tool_use: {'caller': {'type': 'direct'}, 'id': 'toolu_01VvdmmT6sHgERjb7dsKQhS8', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 5}, 'server': False}]

--- finish: tool_calls
--- tool_calls: [ToolCall(id='toolu_014xhQQjAj2fjzHeakyKgw8y', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}), ToolCall(id='toolu_01VvdmmT6sHgERjb7dsKQhS8', name='simple_add', arguments={'a': 10, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}})]


In [ ]:
def delta_text(o):
    "Extract printable content from streaming delta, return None if nothing to print"
    if isinstance(o, Part) and o.type == PartType.tool_result:  return f'🔧 {o.data['name']}\n'
    if isinstance(o,dict): 
        if o.get('thinking'):    return '🧠'
        elif txt:=o.get('text'): return txt
    return None

In [ ]:
chat = AsyncChat('claude-sonnet-4-6', api_name='claude_code', tools=[simple_add])
res = await chat("What is 7+3? Use the tool.", stream=True)
async for o in res: print(delta_text(o) or '', end='')

🧠🧠🔧 simple_add
🧠🧠The result of 7 + 3 is **10**, as returned by the tool. The task was fully complete — nothing further is needed.

In [ ]:
def multiply(a: int, b: int) -> int:
    "Multiply two numbers"
    return a * b

In [ ]:
chat = AsyncChat('claude-sonnet-4-6', api_name='claude_code', tools=[simple_add, multiply])
res = await chat("Calculate 3+5 and 4*6 in parallel using tools.", stream=True, max_steps=5)
async for o in res: print(delta_text(o) or '', end='')

🧠🧠🧠Sure! Both calculations are independent, so I'll run them in parallel:🔧 simple_add
🔧 multiply
🧠🧠🧠Here are the results from both parallel calculations:

| Expression | Result |
|------------|--------|
| 3 + 5      | **8**  |
| 4 × 6      | **24** |

Both operations completed simultaneously, with no need to wait on one before the other.

In [ ]:
res = await chat("What was the last result.", stream=True, max_steps=5)
async for o in res: print(delta_text(o) or '', end='')

🧠🧠🧠The last result was **4 × 6 = 24**.

In [ ]:
chat = AsyncChat('claude-sonnet-4-6', api_name='claude_code', search='l')
res = await chat("Can you search the web for weather in Istanbul", stream=True)
async for o in res: print(delta_text(o) or '', end='')

🧠🔧 WebSearch
🧠🧠🔧 WebFetch
Here's today's weather in **Istanbul** (Thursday, July 23, 2026):

| Condition | Details |
|---|---|
| 🌡️ Temperature | High 25°C / Low 21°C (feels like ~17°C) |
| 🌦️ Conditions | Patchy rain possible, intermittent light showers |
| 🌧️ Precipitation | ~1.5mm, 57% chance of rain |
| 💨 Wind | 25 km/h northerly |
| 💧 Humidity | 64% |
| ☀️ UV Index | High (6.5/11) |
| 🌅 Daylight | 14h 38min (sunrise 05:51, sunset 20:29) |

Notably cooler than the typical July average of ~31°C — a breezy, overcast kind of day with a decent chance of some light rain.

Sources:
- [Today's Weather in Istanbul - Hourly Forecast and Conditions](https://www.easeweather.com/europe/turkey/istanbul/today)
- [Istanbul weather in July 2026 | Weather25](https://www.weather25.com/europe/turkey/istanbul?page=month&month=July)
- [Istanbul, Turkey Monthly Weather | AccuWeather](https://www.accuweather.com/en/tr/istanbul/318251/july-weather/318251)

In [ ]:
res = await chat("What is the weather like again? Just tell me from previous the response", stream=True)
async for o in res: print(delta_text(o) or '', end='')

🧠🧠Here's the Istanbul weather from the previous search (today, July 23, 2026):

| Condition | Details |
|---|---|
| 🌡️ Temperature | High 25°C / Low 21°C (feels like ~17°C) |
| 🌦️ Conditions | Patchy rain possible, intermittent light showers |
| 🌧️ Precipitation | ~1.5mm, 57% chance of rain |
| 💨 Wind | 25 km/h northerly |
| 💧 Humidity | 64% |
| ☀️ UV Index | High (6.5/11) |
| 🌅 Daylight | 14h 38min (sunrise 05:51, sunset 20:29) |

Notably cooler than the typical July average of ~31°C — breezy and overcast with a decent chance of light rain.

Sources:
- [Today's Weather in Istanbul - Hourly Forecast and Conditions](https://www.easeweather.com/europe/turkey/istanbul/today)

In [ ]:
chat.use

total=3,452 | in=3,158 | out=270 | cached=79.4% | cache_new=648 | reasoning=24 | $0.0005 | claude-sonnet-4-6

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()